# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [5]:
# Rutas de los archivos cargados en Google Colab
train_path = Path('/content/train.csv')
contract_path = Path('/content/data_contract.json')

# Cargar los archivos
train_df = pd.read_csv(train_path)

with open(contract_path, 'r', encoding='utf-8') as f:
    contract = json.load(f)

# Identificar las variables predictoras y la variable objetivo
feature_columns = contract['features']
target_column = contract['target']

# Separar características y variable objetivo
X_train = train_df[feature_columns]
y_train = train_df[target_column]

# Mostrar información del dataset
print('X_train:', X_train.shape, '| y_train:', y_train.shape)

print('Clases:')
print(y_train.value_counts().sort_index())

display(X_train.head())

X_train: (142, 13) | y_train: (142,)
Clases:
target
0    47
1    57
2    38
Name: count, dtype: int64


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,13.28,1.64,2.84,15.5,110.0,2.60,2.68,0.34,1.36,4.60,1.09,2.78,880.0
1,13.73,1.50,2.70,22.5,101.0,3.00,3.25,0.29,2.38,5.70,1.19,2.71,1285.0
2,13.39,1.77,2.62,16.1,93.0,2.85,2.94,0.34,1.45,4.80,0.92,3.22,1195.0
3,13.75,1.73,2.41,16.0,89.0,2.60,2.76,0.29,1.81,5.60,1.15,2.90,1320.0
4,13.32,3.24,2.38,21.5,92.0,1.93,0.76,0.45,1.25,8.42,0.55,1.62,650.0


## 2. Pipeline de procesamiento

In [6]:
preprocessor_tree = ColumnTransformer(
    transformers=[('numericas', SimpleImputer(strategy='median'), feature_columns)]
)

preprocessor_scaled = ColumnTransformer(
    transformers=[('numericas', Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), feature_columns)]
)
print('Preprocesadores creados.')

Preprocesadores creados.


**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Porque calcula la media y la desviación estándar de las variables. Si lo ajustamos antes de la validación cruzada, esas estadísticas incluirían datos que deberían estar reservados para la validación. Al incluirlo en el Pipeline, se ajusta solamente con los datos correspondientes al entrenamiento de cada partición, evitando fuga de información.

## 3. Modelos candidatos

In [7]:
models = {
    'DecisionTree': Pipeline(steps=[
        ('preprocessor', preprocessor_tree),
        ('model', DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    'LogisticRegression': Pipeline(steps=[
        ('preprocessor', preprocessor_scaled),
        ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    'KNN': Pipeline(steps=[
        ('preprocessor', preprocessor_scaled),
        ('model', KNeighborsClassifier())
    ])
}
print('Modelos:', ', '.join(models))

Modelos: DecisionTree, LogisticRegression, KNN


## 4. Comparación inicial con validación cruzada

In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
initial_rows = []
for nombre, pipeline in models.items():
    scores = cross_validate(
        pipeline, X_train, y_train, cv=cv,
        scoring='accuracy', return_train_score=True, n_jobs=-1
    )
    initial_rows.append({
        'model': nombre,
        'train_accuracy_mean': scores['train_score'].mean(),
        'val_accuracy_mean': scores['test_score'].mean(),
        'val_accuracy_std': scores['test_score'].std()
    })
initial_comparison = pd.DataFrame(initial_rows).sort_values(
    'val_accuracy_mean', ascending=False
).reset_index(drop=True)
display(initial_comparison.round(4))

,model,train_accuracy_mean,val_accuracy_mean,val_accuracy_std
0,LogisticRegression,1.0000,0.9791,0.0277
1,KNN,0.9771,0.9507,0.0353
2,DecisionTree,1.0000,0.9022,0.0586


## 5. Optimización de hiperparámetros

In [9]:
param_grids = {
    'DecisionTree': {
        'model__max_depth': [2, 3, 5, None],
        'model__min_samples_leaf': [1, 2, 4]
    },
    'LogisticRegression': {
        'model__C': [0.1, 1.0, 10.0]
    },
    'KNN': {
        'model__n_neighbors': [3, 5, 7, 9],
        'model__weights': ['uniform', 'distance']
    }
}

searches = {}
tuned_rows = []
for nombre, pipeline in models.items():
    print('Optimizando:', nombre)
    search = GridSearchCV(
        estimator=pipeline, param_grid=param_grids[nombre],
        scoring='accuracy', cv=cv, n_jobs=-1, refit=True,
        return_train_score=True
    )
    search.fit(X_train, y_train)
    searches[nombre] = search
    idx = search.best_index_
    tuned_rows.append({
        'model': nombre,
        'cv_accuracy_mean': search.best_score_,
        'cv_accuracy_std': search.cv_results_['std_test_score'][idx],
        'train_accuracy_mean': search.cv_results_['mean_train_score'][idx],
        'best_params': search.best_params_
    })

tuned_comparison = pd.DataFrame(tuned_rows).sort_values(
    ['cv_accuracy_mean', 'cv_accuracy_std'], ascending=[False, True]
).reset_index(drop=True)
display(tuned_comparison)

Optimizando: DecisionTree
Optimizando: LogisticRegression
Optimizando: KNN


,model,cv_accuracy_mean,cv_accuracy_std,train_accuracy_mean,best_params
0,LogisticRegression,0.979064,0.027713,1.000000,{'model__C': 1.0}
1,KNN,0.957882,0.040520,0.964773,"{'model__n_neighbors': 7, 'model__weights': 'u..."
2,DecisionTree,0.902463,0.076225,0.985934,"{'model__max_depth': 5, 'model__min_samples_le..."


## 6. Selección y persistencia del campeón

In [10]:
champion_name = tuned_comparison.iloc[0]['model']
champion_search = searches[champion_name]
champion = champion_search.best_estimator_
champion_cv_accuracy = float(champion_search.best_score_)

joblib.dump(champion, ARTIFACTS_DIR / 'champion_model.joblib')

metadata = {
    'champion_name': champion_name,
    'selection_metric': 'accuracy',
    'cv_folds': 5,
    'random_state': RANDOM_STATE,
    'cv_accuracy_mean': champion_cv_accuracy,
    'cv_accuracy_std': float(champion_search.cv_results_['std_test_score'][champion_search.best_index_]),
    'best_params': champion_search.best_params_,
    'features': feature_columns,
    'target': target_column,
    'models_compared': list(models)
}
(ARTIFACTS_DIR / 'training_metadata.json').write_text(
    json.dumps(metadata, indent=4, ensure_ascii=False), encoding='utf-8'
)

# Guardamos todas las combinaciones probadas, no solamente la ganadora.
cv_tables = []
for nombre, search in searches.items():
    tabla = pd.DataFrame(search.cv_results_)[
        ['params', 'mean_train_score', 'mean_test_score', 'std_test_score', 'rank_test_score']
    ].copy()
    tabla.insert(0, 'model', nombre)
    tabla['params'] = tabla['params'].apply(lambda p: json.dumps(p, sort_keys=True))
    cv_tables.append(tabla)
cv_results = pd.concat(cv_tables, ignore_index=True)
cv_results.to_csv(REPORTS_DIR / 'cv_results.csv', index=False)

print('Modelo campeón:', champion_name)
print('Accuracy promedio de CV:', round(champion_cv_accuracy, 4))
print('Mejores hiperparámetros:', champion_search.best_params_)
print('Guardados: champion_model.joblib, training_metadata.json y cv_results.csv')

Modelo campeón: LogisticRegression
Accuracy promedio de CV: 0.9791
Mejores hiperparámetros: {'model__C': 1.0}
Guardados: champion_model.joblib, training_metadata.json y cv_results.csv


In [11]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Se seleccionó el modelo de Regresión Logística porque fue el que presentó un mejor desempeño durante la validación cruzada, alcanzando una precisión promedio cercana al 97,91 %, lo que indica que logra clasificar correctamente la mayoría de los vinos según sus características. Además, es importante considerar la variabilidad de los resultados entre las diferentes particiones, ya que un modelo que mantiene un buen rendimiento en distintos conjuntos de datos genera mayor confianza. En cuanto al sobreajuste, la validación cruzada ayuda a reducir el riesgo de seleccionar un modelo que solamente funcione bien con los datos de entrenamiento, aunque esto no garantiza que no exista sobreajuste. También se tuvo en cuenta la complejidad del modelo, porque la Regresión Logística es relativamente sencilla y no necesita una estructura tan compleja como la de algunos árboles de decisión. Por último, tiene una buena interpretabilidad, ya que permite analizar cómo influyen las diferentes características químicas en la clasificación de los vinos mediante sus coeficientes. Por estas razones se considera una buena opción para utilizarla como modelo final y posteriormente evaluar su desempeño con los datos de prueba en el Challenge 3.

## Checklist

- [ ] Todo el procesamiento está dentro de pipelines.
- [ ] Comparé tres familias de modelos.
- [ ] Optimicé hiperparámetros con CV.
- [ ] Elegí el campeón sin usar test.
- [ ] Guardé el pipeline completo y sus metadatos.

In [14]:
from google.colab import files
from zipfile import ZipFile
from pathlib import Path

# Ruta donde se creará el ZIP
ZIP_PATH = Path('/content/challenge_2_resultados.zip')

# Archivos que necesitamos guardar
archivos = [
    (ARTIFACTS_DIR / 'champion_model.joblib', 'artifacts/champion_model.joblib'),
    (ARTIFACTS_DIR / 'training_metadata.json', 'artifacts/training_metadata.json'),
    (Path('/content/data_contract.json'), 'artifacts/data_contract.json'),
    (REPORTS_DIR / 'cv_results.csv', 'reports/cv_results.csv')
]

# Verificar que todos los archivos existan
for ruta, nombre in archivos:
    if not ruta.exists():
        raise FileNotFoundError(f'No se encontró el archivo: {ruta}')

# Crear el ZIP
with ZipFile(ZIP_PATH, 'w') as zip_file:
    for ruta, nombre in archivos:
        zip_file.write(ruta, nombre)

# Descargar ZIP
files.download(str(ZIP_PATH))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>